In [0]:
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "src")))

from cdc_pipeline.config import BRONZE_TABLE, SILVER_TABLE, GOLD_TABLE
checks = []
bronze_count = spark.table(BRONZE_TABLE).count()
silver_count = spark.table(SILVER_TABLE).count()
gold_count = spark.table(GOLD_TABLE).count()

checks.append(("bronze_not_empty", bronze_count > 0))
checks.append(("silver_not_empty", silver_count > 0))
checks.append(("gold_not_empty", gold_count > 0))

In [0]:
duplicate_group_count = spark.sql(f"""
    SELECT COUNT(*) AS duplicate_group_count
    FROM (
        SELECT order_id
        FROM {SILVER_TABLE}
        GROUP BY order_id
        HAVING COUNT(*) > 1
    ) duplicate_order_ids
""").first()["duplicate_group_count"]

checks.append(("no_duplicate_order_ids", duplicate_group_count == 0))

In [0]:
for name, passed in checks:
    print(f"{'✅' if passed else '❌'} {name}")

failures = [name for name, passed in checks if not passed]
assert not failures, f"Validation failed: {', '.join(failures)}"